# FX quote lift — logistic GD + sklearn (Solution)

**Short name (GitHub):** `FX_Quote_GD_Sklearn`  
Work the **Practice Skeleton** first. This is the worked key for the currency-conversion adaptation of C1_W3 Lab06 + Lab07.


## Inline cheat-sheet

| Item | Result |
|------|--------|
| Gradient check | `dj_db≈0.498618`, `dj_dw≈[0.498333, 0.498839]` |
| GD 10k × α=0.1 | $w\approx(5.281, 5.078)$, $b\approx-14.222$, $J\approx0.017$, fill-rate = 1.0 |
| Spread 1-D | tightness cutoff ≈ 18.7 bps ⇒ quoted-spread cutoff ≈ **15.3 bps** |
| sklearn C=∞ | larger $|w|$, fill-rate 1.0 |
| sklearn L2 | $w\approx(0.90, 0.74)$, fill-rate still 1.0 |


In [ ]:
import copy, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from matplotlib.lines import Line2D

%matplotlib inline
np.set_printoptions(precision=6, suppress=True)
print("Libraries loaded")


## 1. Blotter


In [ ]:
df6 = pd.read_csv("data/fx_quote_blotter6.csv")
X_train = df6[["edge_score", "urgency_score"]].to_numpy(dtype=float)
y_train = df6["lifted"].to_numpy(dtype=float)
print(df6)
print("X_train shape:", X_train.shape)

fig, ax = plt.subplots(figsize=(5.2, 4.3))
ax.scatter(X_train[y_train == 0, 0], X_train[y_train == 0, 1], c="#1f77b4", s=90, label="passed")
ax.scatter(X_train[y_train == 1, 0], X_train[y_train == 1, 1], c="#d62728", marker="x", s=110, label="lifted")
for _, r in df6.iterrows():
    ax.annotate(r["pair"], (r["edge_score"] + 0.06, r["urgency_score"] + 0.06), fontsize=7)
ax.set_xlim(0, 4); ax.set_ylim(0, 3.5)
ax.set_xlabel("Client edge score"); ax.set_ylabel("Urgency score")
ax.set_title("6-quote FX blotter"); ax.legend(frameon=False); ax.grid(True, alpha=0.3)
plt.show()


## 2. Sigmoid, cost, gradient, GD


In [ ]:
def sigmoid(z):
    z = np.clip(np.asarray(z, dtype=float), -50.0, 50.0)
    return 1.0 / (1.0 + np.exp(-z))

def compute_cost_logistic(X, y, w, b):
    m = X.shape[0]
    cost = 0.0
    for i in range(m):
        f = np.clip(sigmoid(np.dot(X[i], w) + b), 1e-15, 1 - 1e-15)
        cost += -(y[i] * np.log(f) + (1 - y[i]) * np.log(1 - f))
    return cost / m

def compute_gradient_logistic(X, y, w, b):
    m, n = X.shape
    dj_dw = np.zeros(n); dj_db = 0.0
    for i in range(m):
        err = sigmoid(np.dot(X[i], w) + b) - y[i]
        for j in range(n):
            dj_dw[j] += err * X[i, j]
        dj_db += err
    return dj_db / m, dj_dw / m

def compute_gradient_logistic_vec(X, y, w, b):
    err = sigmoid(X @ w + b) - y
    return float(np.mean(err)), (X.T @ err) / X.shape[0]

print(sigmoid(np.array([-100.0, 0.0, 100.0])))


In [ ]:
w_tmp = np.array([2., 3.]); b_tmp = 1.
dj_db_tmp, dj_dw_tmp = compute_gradient_logistic(X_train, y_train, w_tmp, b_tmp)
print("dj_db:", dj_db_tmp)
print("dj_dw:", dj_dw_tmp.tolist())
dj_db_v, dj_dw_v = compute_gradient_logistic_vec(X_train, y_train, w_tmp, b_tmp)
print("vec match?", np.allclose(dj_dw_v, dj_dw_tmp) and np.isclose(dj_db_v, dj_db_tmp))


In [ ]:
def gradient_descent(X, y, w_in, b_in, alpha, num_iters, grad_fn=compute_gradient_logistic, verbose=True):
    J_history = []
    w = copy.deepcopy(np.asarray(w_in, dtype=float)); b = float(b_in)
    for i in range(num_iters):
        dj_db, dj_dw = grad_fn(X, y, w, b)
        w = w - alpha * dj_dw; b = b - alpha * dj_db
        J_history.append(compute_cost_logistic(X, y, w, b))
        if verbose and i % math.ceil(num_iters / 10) == 0:
            print(f"Iteration {i:4d}: Cost {J_history[-1]}")
    return w, b, J_history

w_out, b_out, J_history = gradient_descent(X_train, y_train, np.zeros(2), 0.0, 0.1, 10000)
print(f"\nupdated parameters: w:{w_out}, b:{b_out}")
print("final J:", J_history[-1])


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.0, 3.4))
axes[0].plot(J_history[:800], color="#0f766e")
axes[0].set_title("J, first 800 steps"); axes[0].set_xlabel("iter"); axes[0].grid(True, alpha=0.3)
axes[1].plot(np.log10(np.clip(J_history, 1e-12, None)), color="#b45309")
axes[1].set_title("log10 J, 10k steps"); axes[1].set_xlabel("iter"); axes[1].grid(True, alpha=0.3)
fig.tight_layout(); plt.show()


In [ ]:
xx, yy = np.meshgrid(np.linspace(0, 4, 200), np.linspace(0, 3.5, 200))
grid = np.c_[xx.ravel(), yy.ravel()]
prob = sigmoid(grid @ w_out + b_out).reshape(xx.shape)
fig, ax = plt.subplots(figsize=(5.4, 4.4))
cs = ax.contourf(xx, yy, prob, levels=20, cmap="RdBu_r", vmin=0, vmax=1, alpha=0.85)
fig.colorbar(cs, ax=ax, fraction=0.046, label="P(lift)")
ax.contour(xx, yy, prob, levels=[0.5], colors="k", linewidths=1.4)
ax.scatter(X_train[y_train == 0, 0], X_train[y_train == 0, 1], c="#1f77b4", s=90, edgecolor="white", zorder=3, label="passed")
ax.scatter(X_train[y_train == 1, 0], X_train[y_train == 1, 1], c="#d62728", marker="x", s=110, zorder=3, label="lifted")
ax.set_xlim(0, 4); ax.set_ylim(0, 3.5)
ax.set_xlabel("Client edge score"); ax.set_ylabel("Urgency score")
ax.set_title("GD lift boundary"); ax.legend(frameon=False); plt.show()
print("train fill-rate GD:", np.mean((sigmoid(X_train @ w_out + b_out) >= 0.5).astype(float) == y_train))


## 3. sklearn


In [ ]:
lr_default = LogisticRegression()
lr_default.fit(X_train, y_train)
print("Prediction on blotter:", lr_default.predict(X_train))
print("Fill-rate (score):", lr_default.score(X_train, y_train))
print("L2 coef, intercept:", lr_default.coef_, lr_default.intercept_)

lr_unreg = LogisticRegression(C=np.inf, solver="lbfgs", max_iter=2000)
lr_unreg.fit(X_train, y_train)
print("C=inf coef, intercept:", lr_unreg.coef_, lr_unreg.intercept_)

p_un = lr_unreg.predict_proba(grid)[:, 1].reshape(xx.shape)
p_l2 = lr_default.predict_proba(grid)[:, 1].reshape(xx.shape)
fig, ax = plt.subplots(figsize=(5.5, 4.4))
ax.contour(xx, yy, prob, levels=[0.5], colors="#0f766e", linewidths=2)
ax.contour(xx, yy, p_un, levels=[0.5], colors="#1d4ed8", linewidths=1.6, linestyles="--")
ax.contour(xx, yy, p_l2, levels=[0.5], colors="#c2410c", linewidths=1.6, linestyles=":")
ax.scatter(X_train[y_train == 0, 0], X_train[y_train == 0, 1], c="#1f77b4", s=90, edgecolor="white", zorder=3)
ax.scatter(X_train[y_train == 1, 0], X_train[y_train == 1, 1], c="#d62728", marker="x", s=110, zorder=3)
ax.legend(handles=[
    Line2D([0], [0], color="#0f766e", lw=2, label="from-scratch GD"),
    Line2D([0], [0], color="#1d4ed8", lw=1.6, ls="--", label="sklearn C=inf"),
    Line2D([0], [0], color="#c2410c", lw=1.6, ls=":", label="sklearn default L2"),
], frameon=False, fontsize=8)
ax.set_xlim(0, 4); ax.set_ylim(0, 3.5)
ax.set_xlabel("Client edge score"); ax.set_ylabel("Urgency score")
ax.set_title("GD vs sklearn lift boundaries"); plt.show()


## 4. Retail spread (1-D)


In [ ]:
df1 = pd.read_csv("data/fx_spread_1d.csv")
x1 = df1["tightness_bps"].to_numpy(dtype=float)
y1 = df1["converted"].to_numpy(dtype=float)
w1, b1, J1 = gradient_descent(x1.reshape(-1, 1), y1, np.zeros(1), 0.0, 0.1, 8000, verbose=False)
t_cut = float(-b1 / w1[0])
s_cut = 34.0 - t_cut
print("w, b, J:", float(w1[0]), b1, J1[-1])
print("tightness at 50%:", t_cut, "⇒ spread at 50%:", s_cut)

fig, ax = plt.subplots(figsize=(6.2, 3.5))
sp = df1["spread_bps"].to_numpy()
ax.scatter(sp[y1 == 1], y1[y1 == 1], c="#2ca02c", s=80, label="converted")
ax.scatter(sp[y1 == 0], y1[y1 == 0], c="#d62728", marker="x", s=90, label="abandoned")
xs = np.linspace(2, 34, 200)
ax.plot(xs, sigmoid(w1[0] * (34 - xs) + b1), color="black", lw=2, label="P(convert)")
ax.axvline(s_cut, color="#1f77b4", ls=":", label=f"50% ≈ {s_cut:.1f} bps")
ax.set_xlabel("Quoted spread (bps)"); ax.set_ylabel("outcome")
ax.set_ylim(-0.08, 1.08); ax.legend(frameon=False, fontsize=8)
ax.set_title("Wider spread → fewer conversions"); ax.grid(True, alpha=0.3); plt.show()


## 5. More practice


In [ ]:
dfp = pd.read_csv("data/fx_quote_practice.csv")
Xp = dfp[["edge_score", "urgency_score"]].to_numpy(dtype=float)
yp = dfp["lifted"].to_numpy(dtype=float)
wp, bp, Jp = gradient_descent(Xp, yp, np.zeros(2), 0.0, 0.3, 2000, verbose=False)
pred_p = (sigmoid(Xp @ wp + bp) >= 0.5).astype(float)
print("60-quote GD w,b:", wp, bp, "fill-rate:", np.mean(pred_p == yp), "J:", Jp[-1])
lr_p = LogisticRegression(C=np.inf, solver="lbfgs", max_iter=2000)
lr_p.fit(Xp, yp)
print("sklearn C=inf fill-rate:", lr_p.score(Xp, yp))

fig, ax = plt.subplots(figsize=(5.2, 4.2))
ax.scatter(Xp[yp == 0, 0], Xp[yp == 0, 1], c="#1f77b4", s=36, label="passed")
ax.scatter(Xp[yp == 1, 0], Xp[yp == 1, 1], c="#d62728", marker="x", s=44, label="lifted")
xs_line = np.linspace(Xp[:, 0].min() - 0.2, Xp[:, 0].max() + 0.2, 50)
ax.plot(xs_line, -(wp[0] * xs_line + bp) / wp[1], color="#0f766e", lw=1.6, label="GD boundary")
ax.set_xlabel("edge_score"); ax.set_ylabel("urgency_score")
ax.set_title("60-quote stream"); ax.legend(frameon=False, fontsize=8); ax.grid(True, alpha=0.3)
plt.show()


In [ ]:
dfc = pd.read_csv("data/fx_usdcrc_markup.csv")
xc = dfc["usdcrc_markup_pct"].to_numpy(dtype=float)
yc = dfc["converted"].to_numpy(dtype=float)
# convert = 1 when markup is low, so w should be negative if we use raw markup
wc, bc, Jc = gradient_descent(xc.reshape(-1, 1), yc, np.zeros(1), 0.0, 0.8, 8000, verbose=False)
cut = float(-bc / wc[0])
print("USDCRC w,b,J:", float(wc[0]), bc, Jc[-1])
print("P(convert)=0.5 at markup % ≈", cut)
print("train acc:", np.mean(((sigmoid(wc[0] * xc + bc) >= 0.5).astype(float) == yc)))
print(
    "San José retail desk: on this toy cart the 50% conversion point sits near "
    f"{cut:.2f}% markup. Tighten displayed USD/CRC toward that band if the goal is fill, "
    "widen it if the goal is margin — this is a teaching cutoff, not a posted rate."
)


In [ ]:
w_alt, b_alt, _ = gradient_descent(
    X_train, y_train, np.zeros(2), 0.0, 0.1, 10000,
    grad_fn=compute_gradient_logistic_vec, verbose=False,
)
print("alt w,b:", w_alt, b_alt)
print("match 6-quote GD?", np.allclose(w_alt, w_out) and np.isclose(b_alt, b_out))


## 6. Simulation


In [ ]:
ALPHAS = [0.01, 0.05, 0.1, 0.5, 1.0]
ITERS_ALPHA = 1500
SIZES = [20, 40, 80, 160]
N_REPS = 20
NOISE = 0.06
ALPHA_MC = 0.3
ITERS_MC = 400
SEED = 11
rng = np.random.default_rng(SEED)

fig, axes = plt.subplots(1, 2, figsize=(9.4, 3.6))
for a in ALPHAS:
    _, _, Jh = gradient_descent(X_train, y_train, np.zeros(2), 0.0, a, ITERS_ALPHA, verbose=False)
    axes[0].plot(Jh, label=f"α={a}")
axes[0].set_title("J vs α on the 6-quote blotter")
axes[0].set_xlabel("iteration"); axes[0].set_ylabel("J")
axes[0].legend(fontsize=8, frameon=False); axes[0].grid(True, alpha=0.3)

acc_mean, acc_std = [], []
for m in SIZES:
    accs = []
    for _ in range(N_REPS):
        half = m // 2
        Xn = np.vstack([
            rng.normal(loc=[1.1, 1.0], scale=0.5, size=(half, 2)),
            rng.normal(loc=[2.9, 2.6], scale=0.5, size=(m - half, 2)),
        ])
        yn = np.array([0] * half + [1] * (m - half), dtype=float)
        flip = rng.random(m) < NOISE
        yn = np.where(flip, 1 - yn, yn)
        wh, bh, _ = gradient_descent(Xn, yn, np.zeros(2), 0.0, ALPHA_MC, ITERS_MC, verbose=False)
        accs.append(np.mean((sigmoid(Xn @ wh + bh) >= 0.5).astype(float) == yn))
    acc_mean.append(np.mean(accs)); acc_std.append(np.std(accs))

axes[1].errorbar(SIZES, acc_mean, yerr=acc_std, marker="o", color="#0f766e", capsize=3)
axes[1].set_title(f"Fill-rate vs blotter size (last-look noise={NOISE})")
axes[1].set_xlabel("quotes"); axes[1].set_ylabel("mean ± sd acc")
axes[1].set_ylim(0.6, 1.02); axes[1].grid(True, alpha=0.3)
fig.tight_layout(); plt.show()
print("mean fill-rate by m:", list(zip(SIZES, acc_mean)))


## 7. Audience rewrite

**Pricing quant.** Batch GD on the six-quote blotter with $\alpha=0.1$ and 10k steps reaches $w\approx(5.28,5.08)$, $b\approx-14.22$, $J\approx0.017$. The loop gradient matches the published Lab06 digits; $X^\top(f-y)/m$ is identical. sklearn with $C=\infty$ also fills every toy quote but stops at a different scale of $w$. Default $C=1$ L2 shrinks $|w|$ by an order of magnitude. Do not treat coefficient equality as a regression test against the pricing library.

**Sales dealer / ops.** Read $f$ as P(lift). The black 0.5 line is the quotes we expect the client to take given edge and urgency. On the retail spread drill the 50% conversion point sits near **15 bps**. On the USD/CRC cart it sits near **1.7% markup**. Tightening the quote raises fill and cuts margin; widening does the opposite. Last-look and stale quotes show up in the simulation as flipped labels — they cap how clean the fill-rate can look.

**Treasury executive.** Two estimators classify all six training quotes correctly. The library default quietly shrinks weights (regularization), which is usually what you want on a live book and is why the printed formulas disagree. Six rows and 100% in-sample fill are not a mandate to auto-price. Next control is a time-split on real streams, plus an explicit last-look policy.

**Retail converter.** If the app shows a fairer rate and you really need the colones this afternoon, you are more likely to tap Convert. The model is just a smooth version of that idea: it draws a line between “people who converted” and “people who closed the screen,” then inches the line until it sits between the two groups. A wider hidden markup makes more people close the screen.


## 8. Takeaways

- Same math as Lab06/07; the labels are lift / convert instead of tumor class.
- Clip $z$ and $f$. If $J$ rises, cut $\alpha$.
- sklearn default ≠ unregularized GD. Compare fill-rate and the boundary.
- Threshold = sales appetite; $C$ = smoothness; last-look noise = data quality.
- Teaching blotter only — not a live auto-pricer and not FX advice.
